# 🔭 GSoC 2026 — ML4SCI DeepLense | Specific Test V: Gravitational Lens Finding

**Author:** Yash Yadav  
**GitHub:** [github.com/Yash1085](https://github.com/Yash1085)  
**Kaggle:** [kaggle.com/yash2072005](https://www.kaggle.com/yash2072005)  
**LinkedIn:** [linkedin.com/in/yash-yadav007](https://www.linkedin.com/in/yash-yadav007/)  
**Organization:** ML4SCI — DeepLense  

---

## Task
Binary classification — identify strong gravitational lenses from non-lensed galaxies using multi-filter observational imaging data.

## Dataset
- 3-channel (multi-filter) `.npy` arrays, shape `(3, 64, 64)`
- **Train:** 1,730 lenses vs 28,675 non-lenses → imbalance ratio **1 : 16.6**
- **Test:** 195 lenses vs 19,455 non-lenses → imbalance ratio **1 : 99.7**
- Primary evaluation metric: **ROC-AUC** | Secondary: **PR-AUC**, **TPR @ FPR = 1%**

---

## Strategy Overview

| Challenge | Approach |
|---|---|
| Extreme class imbalance (1:100 at test) | **Standard BCE + WeightedRandomSampler** + threshold calibration |
| Small 64×64 inputs | EfficientNet-B0 pretrained backbone (optimal compound scaling) |
| Physical rotational symmetry | Dihedral augmentations + **$C_8$ (Continuous $SO(2)$ approx.) Equivariant Network** |
| Train/test imbalance mismatch | Probability calibration + threshold sweep at test-time ratio |
| Validation reliability | 5-Fold Stratified Cross-Validation + **Ensemble Inference** |
| Interpretability | Grad-CAM on TP / FP / FN samples |
| Free inference boost | 5-pass Test-Time Augmentation (TTA) with **Max-Pooling Aggregation** |

---

## Pipeline Sections
1. Imports & Reproducibility  
2. Configuration  
3. Data Loading & Per-Channel Statistics  
4. Exploratory Data Analysis  
5. Transforms & Augmentation  
6. Dataset & DataLoaders  
7. Focal Loss  
8. Model — EfficientNet-B0  
9. Optimiser & Scheduler  
10. Training Loop with Early Stopping  
11. Training Curves  
12. Threshold Calibration on Validation Set  
13. Test Inference with TTA  
14. Full Evaluation — ROC, PR, Confusion Matrix, **TPR @ FPR=1%**  
15. **5-Fold Stratified Cross-Validation**  
16. **Grad-CAM Interpretability**  
17. **D₄ Equivariant Neural Network Baseline**  
18. Final Results Summary  


## 1. Imports & Reproducibility

In [ ]:
import os, glob, random, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, Subset
from torch.cuda.amp import autocast, GradScaler
from torchvision import models, transforms
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (
    roc_curve, auc, roc_auc_score, average_precision_score,
    precision_recall_curve, confusion_matrix, ConfusionMatrixDisplay,
    classification_report
)
from sklearn.calibration import CalibratedClassifierCV
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

# ── Device ────────────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


## 2. Configuration

All hyperparameters in one place for easy reproducibility.

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_PATH      = '/kaggle/input/datasets/yash2072005/gravitational-lens-finding'
TRAIN_LENS_DIR = os.path.join(BASE_PATH, 'train_lenses')
TRAIN_NONL_DIR = os.path.join(BASE_PATH, 'train_nonlenses')
TEST_LENS_DIR  = os.path.join(BASE_PATH, 'test_lenses')
TEST_NONL_DIR  = os.path.join(BASE_PATH, 'test_nonlenses')
CKPT_PATH      = '/kaggle/working/best_effnet.pth'
ENN_CKPT_PATH  = '/kaggle/working/best_enn.pth'

# ── Training ──────────────────────────────────────────────────────────────────
BATCH_SIZE   = 128
NUM_EPOCHS   = 40
LR           = 1e-4
WEIGHT_DECAY = 1e-4
VAL_FRAC     = 0.15
EARLY_STOP_P = 15 
NUM_WORKERS  = 0

# ── TTA ───────────────────────────────────────────────────────────────────────
TTA_STEPS    = 5

# ── CV ────────────────────────────────────────────────────────────────────────
CV_FOLDS     = 5
CV_EPOCHS    = 30
CV_PATIENCE  = 5

print('Config loaded.')

## 3. Data Loading & Per-Channel Statistics

Per-channel mean/std are computed **from the training set only** to prevent
data leakage into validation and test sets.  
Per-channel (not per-image) normalisation preserves the physically meaningful
relative flux information across the three observational filters.


In [ ]:
def collect_files(lens_dir, nonlens_dir):
    lens_f    = sorted(glob.glob(os.path.join(lens_dir,    '*.npy')))
    nonlens_f = sorted(glob.glob(os.path.join(nonlens_dir, '*.npy')))
    files  = lens_f + nonlens_f
    labels = [1]*len(lens_f) + [0]*len(nonlens_f)
    print(f'  Lenses: {len(lens_f):>6,}  |  Non-lenses: {len(nonlens_f):>6,}'
          f'  |  Ratio 1:{len(nonlens_f)/max(len(lens_f),1):.1f}')
    return files, labels

print('Train:'); train_files, train_labels = collect_files(TRAIN_LENS_DIR, TRAIN_NONL_DIR)
print('Test: '); test_files,  test_labels  = collect_files(TEST_LENS_DIR,  TEST_NONL_DIR)

# ── Per-channel stats from training data only ─────────────────────────────────
print('\nComputing per-channel normalisation stats from training set...')
arrays     = [np.load(f).astype(np.float32) for f in tqdm(train_files, leave=False)]
train_stack = np.stack(arrays, axis=0)                  # (N, 3, 64, 64)
TRAIN_MEAN  = train_stack.mean(axis=(0,2,3)).tolist()
TRAIN_STD   = [max(s, 1e-8) for s in train_stack.std(axis=(0,2,3)).tolist()]
del train_stack, arrays

print(f'Channel means : {[f"{m:.4f}" for m in TRAIN_MEAN]}')
print(f'Channel stds  : {[f"{s:.4f}" for s in TRAIN_STD]}')


## 4. Exploratory Data Analysis

Visual inspection of sample lenses and non-lenses across all three filters.  
Key observation: lenses show faint **Einstein ring arcs** around a bright central
deflector — a signal that is easily overwhelmed by the non-lens majority.


In [ ]:
def show_samples(files, labels, title, n=5):
    lens_idx    = [i for i,l in enumerate(labels) if l==1][:n]
    nonlens_idx = [i for i,l in enumerate(labels) if l==0][:n]

    fig, axes = plt.subplots(2*3, n, figsize=(3*n, 6*2))
    filter_names = ['Filter g', 'Filter r', 'Filter i']

    for row_group, (indices, group_label) in enumerate(
            [(lens_idx, 'LENS'), (nonlens_idx, 'NON-LENS')]):
        for ch in range(3):
            row = row_group*3 + ch
            for col, idx in enumerate(indices):
                img = np.load(files[idx]).astype(np.float32)
                ax  = axes[row, col]
                ax.imshow(img[ch], cmap='hot', origin='lower',
                          vmin=np.percentile(img[ch],1),
                          vmax=np.percentile(img[ch],99))
                if col == 0:
                    ax.set_ylabel(f'{group_label}\n{filter_names[ch]}', fontsize=8)
                ax.axis('off')

    fig.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('eda_samples.png', dpi=120, bbox_inches='tight')
    plt.show()

show_samples(train_files, train_labels, 'Sample Images — All Three Filters')


## 5. Transforms & Augmentation

**Augmentation rationale:**
- Gravitational lensing images have no preferred orientation →
  full dihedral group D₄ augmentations (flips + 90° rotations) are physically valid
- PSF-blur augmentation simulates telescope point-spread function variation
- Gaussian noise augmentation simulates shot noise in low-S/N observations
- No colour jitter — inter-channel flux ratios are physically meaningful

Test/validation transforms apply only normalisation (no augmentation = no leakage).


In [ ]:
normalize = transforms.Normalize(mean=TRAIN_MEAN, std=TRAIN_STD)

train_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomApply([transforms.RandomRotation((90,90))],   p=0.5),
    transforms.RandomApply([transforms.RandomRotation((180,180))], p=0.25),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.5,1.5))], p=0.3),
    normalize,
])

val_transforms = transforms.Compose([normalize])

print('Transforms defined.')
print(f'  Train: {len(train_transforms.transforms)} stages')
print(f'  Val  : {len(val_transforms.transforms)} stages')


## 6. Dataset, Stratified Split & DataLoaders

- **Stratified 15% validation split** ensures the lens/non-lens ratio is
  preserved in both train and val subsets
- **WeightedRandomSampler** over-samples lenses in each training batch so every
  batch contains a balanced mix — essential when focal loss alone is insufficient


In [ ]:
class AstroLensDataset(Dataset):
    def __init__(self, filepaths, labels, transform=None):
        self.filepaths = filepaths
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.filepaths)

    def __getitem__(self, idx):
        img   = torch.from_numpy(np.load(self.filepaths[idx]).astype(np.float32))
        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        if self.transform:
            img = self.transform(img)
        return img, label


# ── Stratified split ──────────────────────────────────────────────────────────
tr_idx, val_idx = train_test_split(
    np.arange(len(train_files)),
    test_size=VAL_FRAC,
    random_state=SEED,
    stratify=train_labels,
)

tr_files  = [train_files[i]  for i in tr_idx]
tr_labels = [train_labels[i] for i in tr_idx]
vl_files  = [train_files[i]  for i in val_idx]
vl_labels = [train_labels[i] for i in val_idx]

print(f'Train subset : {len(tr_files):,}  '
      f'(+:{sum(tr_labels)}  -:{sum(1 for l in tr_labels if l==0)})')
print(f'Val   subset : {len(vl_files):,}  '
      f'(+:{sum(vl_labels)}  -:{sum(1 for l in vl_labels if l==0)})')

# ── WeightedRandomSampler ─────────────────────────────────────────────────────
tr_labels_arr = np.array(tr_labels)
class_counts  = np.bincount(tr_labels_arr)
weights       = 1.0 / class_counts[tr_labels_arr]
sampler       = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)

# ── DataLoaders ───────────────────────────────────────────────────────────────
train_ds = AstroLensDataset(tr_files,  tr_labels,   train_transforms)
val_ds   = AstroLensDataset(vl_files,  vl_labels,   val_transforms)
test_ds  = AstroLensDataset(test_files, test_labels, val_transforms)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f'\nTrain batches : {len(train_loader)}')
print(f'Val   batches : {len(val_loader)}')
print(f'Test  batches : {len(test_loader)}')


## 7. Focal Loss

Standard `BCEWithLogitsLoss` with `pos_weight` assigns equal difficulty weight
to every non-lens sample. **Focal Loss** (Lin et al., 2017) adds a modulating
factor $(1 - p_t)^\gamma$ that down-weights easy negatives, forcing the model
to focus on hard, ambiguous examples.

$$\mathcal{L}_{\text{focal}} = -\alpha_t (1 - p_t)^\gamma \log(p_t)$$

- $\alpha = 0.95$ : upweights the rare positive class  
- $\gamma = 2.0$  : standard value; higher values increase focus on hard examples

In [ ]:
FOCAL_GAMMA = 2.0
class FocalLoss(nn.Module):
    def __init__(self, gamma: float = 2.0):
        super().__init__()
        self.gamma = gamma

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        bce   = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        probs = torch.sigmoid(logits)
        p_t   = probs * targets + (1 - probs) * (1 - targets)
        loss  = (1 - p_t) ** self.gamma * bce
        return loss.mean()


criterion = FocalLoss(gamma=FOCAL_GAMMA)
print(f'Alpha-Stripped FocalLoss  gamma={FOCAL_GAMMA}')
print(' No alpha term — class balance handled by WeightedRandomSampler')

## 8. Model — EfficientNet-B0 (Fine-tuned)

**Why EfficientNet-B0?**
- Compound scaling balances depth/width/resolution optimally for 64×64 inputs
- ImageNet pretrained weights provide strong low-level feature priors
  (edge detectors, texture filters) that transfer well to astrophysical images
- ~5.3M parameters — powerful enough for this task, compact enough to prevent
  overfitting on 1,730 positive samples
- Outperforms custom SE-ResNets of similar depth on this resolution regime

**Head design:**
- Dropout(0.4) → prevents co-adaptation of the final features
- Single linear output → compatible with Focal Loss (raw logit)


In [ ]:
class LensFinderEfficientNet(nn.Module):
    def __init__(self, dropout: float = 0.4):
        super().__init__()
        backbone = models.efficientnet_b0(weights='IMAGENET1K_V1')
        in_feats = backbone.classifier[1].in_features
        backbone.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(in_feats, 1)
        )
        self.net = backbone

    def forward(self, x):
        return self.net(x).squeeze(1)   # (B,)


model = LensFinderEfficientNet().to(device)
total  = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params    : {total:,}')
print(f'Trainable params: {trainable:,}')

# quick forward pass check
with torch.no_grad():
    dummy = torch.randn(4, 3, 64, 64).to(device)
    out   = model(dummy)
print(f'Output shape    : {out.shape}  ✓')


## 9. Optimiser & Scheduler

| Component | Choice | Rationale |
|---|---|---|
| Optimiser | AdamW | Decoupled weight decay, robust to LR scale |
| LR | 1e-4 | Conservative start for fine-tuning pretrained weights |
| Scheduler | CosineAnnealingWarmRestarts | Warm restarts escape local minima; T₀=10 gives 4 complete cycles in 40 epochs |
| AMP | GradScaler + autocast | ~2× training speedup on T4/V100 with no accuracy loss |


In [ ]:
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=10, T_mult=1, eta_min=1e-6
)
scaler = GradScaler()
print('Optimiser : AdamW')
print('Scheduler : CosineAnnealingWarmRestarts  T_0=10')
print('AMP       : GradScaler + autocast  ✓')


## 10. Training Loop with Validation & Early Stopping

- Model is evaluated on the validation set after every epoch
- **Best checkpoint** is saved by validation ROC-AUC (not loss)
- **Early stopping** with patience=7 prevents overfitting while allowing
  learning rate warm restarts to recover from temporary dips
- Both train and val metrics are logged for curve analysis


In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_probs, all_labels = [], []
    total_loss = 0.0
    for imgs, lbls in loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        with autocast():
            logits = model(imgs)
            loss   = criterion(logits, lbls)
        total_loss += loss.item()
        all_probs.extend(torch.sigmoid(logits).cpu().numpy())
        all_labels.extend(lbls.cpu().numpy())
    probs  = np.array(all_probs)
    labels = np.array(all_labels)
    roc    = roc_auc_score(labels, probs)
    pr     = average_precision_score(labels, probs)
    return total_loss / len(loader), roc, pr


history = {'train_loss':[], 'val_loss':[], 'val_roc':[], 'val_pr':[], 'lr':[]}
best_val_pr    = 0.0  
patience_count = 0
best_ckpt      = None

print(f'Training EfficientNet-B0  [{NUM_EPOCHS} epochs max]')
print('='*65)

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    ep_loss = 0.0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        with autocast():
            loss = criterion(model(imgs), lbls)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        ep_loss += loss.item()
    scheduler.step()

    tr_loss = ep_loss / len(train_loader)
    vl_loss, vl_roc, vl_pr = evaluate(model, val_loader)
    cur_lr  = optimizer.param_groups[0]['lr']

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['val_roc'].append(vl_roc)
    history['val_pr'].append(vl_pr)
    history['lr'].append(cur_lr)

    flag = ''
    if vl_pr > best_val_pr:
        best_val_pr    = vl_pr
        patience_count = 0
        best_ckpt = {'epoch': epoch, 'model_state_dict': model.state_dict(),
                     'val_roc': vl_roc, 'val_pr': vl_pr}
        torch.save(best_ckpt, CKPT_PATH)
        flag = '  ✦ best'
    else:
        patience_count += 1

    if epoch % 5 == 0 or epoch == 1 or flag:
        print(f'Ep {epoch:>3}/{NUM_EPOCHS}  '
              f'tr_loss={tr_loss:.4f}  vl_loss={vl_loss:.4f}  '
              f'val_AUC={vl_roc:.4f}  val_PR={vl_pr:.4f}  '
              f'lr={cur_lr:.2e}{flag}')

    if patience_count >= EARLY_STOP_P:
        print(f'\nEarly stop at epoch {epoch}.  Best val PR = {best_val_pr:.4f}')
        break

print(f'\nBest checkpoint: epoch={best_ckpt["epoch"]}  '
      f'val_AUC={best_ckpt["val_roc"]:.4f}  val_PR={best_ckpt["val_pr"]:.4f}')

## 11. Training Curves

In [ ]:
best_val_auc = max(history['val_roc']) 

ep = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 4, figsize=(22, 4))
fig.suptitle('Training Diagnostics — EfficientNet-B0', fontsize=12)

axes[0].plot(ep, history['train_loss'], label='Train', color='steelblue')
axes[0].plot(ep, history['val_loss'],   label='Val',   color='tomato')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(ep, history['val_roc'], color='purple')
axes[1].axhline(best_val_auc, ls='--', lw=0.8, color='gray',
                label=f'Best={best_val_auc:.4f}')
axes[1].set_title('Val ROC-AUC'); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(ep, history['val_pr'], color='darkorange')
axes[2].set_title('Val PR-AUC'); axes[2].grid(alpha=0.3)

axes[3].plot(ep, history['lr'], color='green')
axes[3].set_title('Learning Rate'); axes[3].grid(alpha=0.3)
axes[3].set_yscale('log')

for ax in axes:
    ax.set_xlabel('Epoch')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()


## 12. Threshold Calibration on Validation Set

Because our training batches are artificially balanced (1:1) via the `WeightedRandomSampler`, the model's raw probability outputs are calibrated for a balanced world. However, the validation and test sets feature severe imbalances (1:16.6 and 1:100). 

As a result, a default 0.5 decision threshold is mathematically incorrect and will yield massive False Positives. We must sweep the decision thresholds across the true imbalanced validation set to find the optimal operating point that maximizes the F1-Score.

In [ ]:
# Add this import at the top of the cell
from sklearn.metrics import f1_score

# ── Load best checkpoint ──────────────────────────────────────────────────────
ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

# ── Collect val probabilities ─────────────────────────────────────────────────
val_probs, val_targs = [], []
with torch.no_grad():
    for imgs, lbls in val_loader:
        logits = model(imgs.to(device))
        val_probs.extend(torch.sigmoid(logits).cpu().numpy())
        val_targs.extend(lbls.numpy())

val_probs = np.array(val_probs)
val_targs = np.array(val_targs)

# ── Threshold sweep (Expanded bounds due to sampler shift) ────────────────────

thresholds = np.linspace(0.01, 0.99, 100)
f1_scores  = [f1_score(val_targs, (val_probs >= t).astype(int)) for t in thresholds]

OPTIMAL_THRESH = thresholds[np.argmax(f1_scores)]
print(f'Optimal Threshold (Max F1) : {OPTIMAL_THRESH:.4f}')
print(f'Max Validation F1-Score    : {np.max(f1_scores):.4f}')

# ── Plot Calibration Curve ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(thresholds, f1_scores, color='purple', lw=2, label='F1 Score')
ax.axvline(OPTIMAL_THRESH, color='red', ls='--', lw=1.5, 
           label=f'Optimal: {OPTIMAL_THRESH:.4f}')
ax.set_xlabel('Threshold'); ax.set_ylabel('F1 Score')
ax.set_title('Threshold vs F1 (Validation Set)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('threshold_calibration.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Test-Time Augmentation (TTA) Inference

TTA averages predictions over `TTA_STEPS=5` independently augmented views of
each test image. Because lensing images have physical rotational symmetry,
augmenting with flips and 90° rotations at test time is information-preserving.
This typically yields +0.2–0.5% AUC improvement at zero training cost.


In [ ]:
tta_aug = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomApply([transforms.RandomRotation((90,90))], p=0.5),
    normalize,
])

@torch.no_grad()
def predict_tta(model, files, labels, n_steps=TTA_STEPS):
    model.eval()
    all_step_probs = []
    
    for step in range(n_steps):
        step_ds = AstroLensDataset(files, labels, tta_aug)
        loader  = DataLoader(step_ds, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=True)
        step_probs = []
        for imgs, _ in loader:
            with autocast():
                logits = model(imgs.to(device))
            step_probs.extend(torch.sigmoid(logits).cpu().numpy())
        all_step_probs.append(step_probs)

    probs = np.max(np.array(all_step_probs), axis=0) 
    labels_arr = np.array(labels)
    return probs, labels_arr

print(f'Running {TTA_STEPS}-pass TTA on test set (Max-Pooling Aggregation)...')
test_probs_tta, test_targets = predict_tta(model, test_files, test_labels)
print(f'Done. test_probs_tta shape: {test_probs_tta.shape}')

## 14. Full Evaluation — ROC, PR Curves, Confusion Matrix & TPR @ FPR = 1%

### Why TPR @ FPR = 1%?
In real wide-field surveys (HSC-SSP, LSST), the lens density is so low that
even a 1% false-positive rate generates thousands of spurious candidates that
overwhelm expert visual inspection. The **True Positive Rate at FPR = 1%**
is therefore the standard operating-point metric in the astrophysics
lens-finding literature (Lanusse et al. 2018, Canameras et al. 2020).


In [ ]:
# ── Helper: TPR at target FPR ─────────────────────────────────────────────────
def tpr_at_fpr(y_true, y_scores, target_fpr=0.01):
    fpr, tpr, thresholds = roc_curve(y_true, y_scores)
    valid = np.where(fpr <= target_fpr)[0]
    if len(valid) == 0:
        return 0.0, thresholds[0]
    idx = valid[-1]
    return float(tpr[idx]), float(thresholds[idx])

# ── Compute all metrics ───────────────────────────────────────────────────────
final_probs = test_probs_tta
final_preds = (final_probs >= OPTIMAL_THRESH).astype(int)

roc_auc  = roc_auc_score(test_targets, final_probs)
pr_auc   = average_precision_score(test_targets, final_probs)
tpr_1pct, thresh_1pct = tpr_at_fpr(test_targets, final_probs, target_fpr=0.01)
tpr_5pct, _           = tpr_at_fpr(test_targets, final_probs, target_fpr=0.05)

print('='*60)
print('  FINAL TEST SET RESULTS — EfficientNet-B0 + TTA')
print('='*60)
print(f'  ROC-AUC          : {roc_auc:.4f}')
print(f'  PR-AUC           : {pr_auc:.4f}')
print(f'  TPR @ FPR = 1%   : {tpr_1pct:.4f}  (threshold={thresh_1pct:.4f})')
print(f'  TPR @ FPR = 5%   : {tpr_5pct:.4f}')
print(f'  Decision thresh  : {OPTIMAL_THRESH:.4f}  (F1-optimal on val set)')
print('='*60)
print(classification_report(test_targets.astype(int), final_preds,
                             target_names=['Non-Lens', 'Lens'], digits=4))

# ── Plots ─────────────────────────────────────────────────────────────────────
fpr_arr, tpr_arr, _ = roc_curve(test_targets, final_probs)
prec_arr, rec_arr, _ = precision_recall_curve(test_targets, final_probs)

fig = plt.figure(figsize=(20, 5))
gs  = gridspec.GridSpec(1, 4, figure=fig)

# Full ROC
ax0 = fig.add_subplot(gs[0])
ax0.plot(fpr_arr, tpr_arr, lw=1.8, color='steelblue', label=f'AUC={roc_auc:.4f}')
ax0.plot([0,1],[0,1],'--',color='gray',lw=0.8)
ax0.set_xlabel('FPR'); ax0.set_ylabel('TPR')
ax0.set_title('ROC Curve'); ax0.legend(); ax0.grid(alpha=0.3)

# Zoomed ROC — astrophysics operating region
ax1 = fig.add_subplot(gs[1])
mask = fpr_arr <= 0.05
ax1.plot(fpr_arr[mask], tpr_arr[mask], lw=2, color='steelblue')
ax1.axvline(0.01, color='crimson', lw=1.2, ls='--',
            label=f'FPR=1%  TPR={tpr_1pct:.3f}')
valid_idx = np.where(fpr_arr <= 0.01)[0]
if len(valid_idx):
    ax1.scatter([fpr_arr[valid_idx[-1]]], [tpr_1pct], color='crimson', s=60, zorder=5)
ax1.set_xlabel('FPR'); ax1.set_ylabel('TPR')
ax1.set_title('ROC — zoomed FPR ≤ 5%'); ax1.legend(); ax1.grid(alpha=0.3)

# PR Curve
ax2 = fig.add_subplot(gs[2])
ax2.plot(rec_arr, prec_arr, lw=1.8, color='darkorange', label=f'PR-AUC={pr_auc:.4f}')
ax2.set_xlabel('Recall'); ax2.set_ylabel('Precision')
ax2.set_title('Precision-Recall Curve'); ax2.legend(); ax2.grid(alpha=0.3)

# Confusion Matrix
ax3 = fig.add_subplot(gs[3])
cm = confusion_matrix(test_targets.astype(int), final_preds)
disp = ConfusionMatrixDisplay(cm, display_labels=['Non-Lens','Lens'])
disp.plot(ax=ax3, colorbar=False, cmap='Blues')
ax3.set_title(f'Confusion Matrix\n(thresh={OPTIMAL_THRESH:.3f})')

fig.suptitle('EfficientNet-B0 — Full Test Evaluation', fontsize=13)
plt.tight_layout()
plt.savefig('full_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()


## 15. Error Analysis — False Negatives & False Positives

Understanding *which* objects the model fails on is as important as the
aggregate metrics. False negatives (missed lenses) and false positives
(contaminants classified as lenses) reveal model limitations and guide
Phase 3 contaminant analysis in the GSoC project.


In [ ]:
test_targets_int = test_targets.astype(int)
fn_mask = (test_targets_int == 1) & (final_preds == 0)
fp_mask = (test_targets_int == 0) & (final_preds == 1)
tp_mask = (test_targets_int == 1) & (final_preds == 1)

fn_files  = [test_files[i]  for i in np.where(fn_mask)[0]]
fn_scores = final_probs[fn_mask]
fp_files  = [test_files[i]  for i in np.where(fp_mask)[0]]
fp_scores = final_probs[fp_mask]
tp_files  = [test_files[i]  for i in np.where(tp_mask)[0]]
tp_scores = final_probs[tp_mask]

print(f'True  Positives : {tp_mask.sum():>4}')
print(f'False Positives : {fp_mask.sum():>4}  ← contaminants (spiral/ring galaxies)')
print(f'False Negatives : {fn_mask.sum():>4}  ← missed lenses (faint arcs)')
print(f'True  Negatives : {(~fn_mask & ~fp_mask & (test_targets_int==0)).sum():>4}')

def plot_error_grid(files, scores, title, n=6, cmap='hot'):
    n = min(n, len(files))
    if n == 0:
        print(f'No samples for: {title}'); return
    fig, axes = plt.subplots(1, n, figsize=(3*n, 3.2))
    if n == 1: axes = [axes]
    # sort by score (most confident first for TP/FP, least confident for FN)
    order = np.argsort(scores)[::-1] if 'Positive' in title else np.argsort(scores)
    for col, idx in enumerate(order[:n]):
        img = np.load(files[idx]).astype(np.float32)
        ax  = axes[col]
        ax.imshow(img[0], cmap=cmap, origin='lower',
                  vmin=np.percentile(img[0],2), vmax=np.percentile(img[0],98))
        ax.set_title(f'p={scores[idx]:.3f}', fontsize=9)
        ax.axis('off')
    fig.suptitle(title, fontsize=11, fontweight='bold')
    plt.tight_layout()
    fname = title.lower().replace(' ','_').replace('/','_') + '.png'
    plt.savefig(fname, dpi=120, bbox_inches='tight'); plt.show()

plot_error_grid(tp_files, tp_scores, 'True Positives — Correctly Found Lenses')
plot_error_grid(fp_files, fp_scores, 'False Positives — Contaminants (Key Challenge)')
plot_error_grid(fn_files, fn_scores, 'False Negatives — Missed Lenses (Faint Arcs)')


## 16. 5-Fold Stratified Cross-Validation

A single validation split on 1,730 positives (~260 val positives) has high
variance. 5-fold stratified CV gives stable, trustworthy estimates of
generalisation performance.

**Protocol:**
- 5 independent models trained from scratch with different val folds
- Out-of-fold (OOF) predictions assembled into a full pseudo-test set
- Final metrics computed both per-fold (mean ± std) and on the OOF aggregate
- Best model per fold is saved for ensemble inference


In [ ]:
skf     = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
labels_arr_full = np.array(train_labels)
indices_full    = np.arange(len(train_files))

fold_roc_list, fold_pr_list, fold_tpr_list = [], [], []
oof_scores_arr  = np.zeros(len(train_files))
oof_labels_arr  = labels_arr_full.copy()

print(f'5-Fold Stratified CV  |  {CV_FOLDS} folds × {CV_EPOCHS} epochs max')
print('='*65)

for fold, (tr_idx_cv, val_idx_cv) in enumerate(
        skf.split(indices_full, labels_arr_full)):

    print(f'\n── Fold {fold+1}/{CV_FOLDS} ───────────────────────────────────────')

    # datasets
    tr_f_cv  = [train_files[i]  for i in tr_idx_cv]
    tr_l_cv  = [train_labels[i] for i in tr_idx_cv]
    vl_f_cv  = [train_files[i]  for i in val_idx_cv]
    vl_l_cv  = [train_labels[i] for i in val_idx_cv]

    cv_counts = np.bincount(np.array(tr_l_cv))
    cv_w      = 1.0 / cv_counts[np.array(tr_l_cv)]
    cv_samp   = WeightedRandomSampler(cv_w, len(cv_w), replacement=True)

    cv_tr_ds  = AstroLensDataset(tr_f_cv, tr_l_cv, train_transforms)
    cv_vl_ds  = AstroLensDataset(vl_f_cv, vl_l_cv, val_transforms)
    cv_tr_ld  = DataLoader(cv_tr_ds, batch_size=BATCH_SIZE, sampler=cv_samp,
                           num_workers=NUM_WORKERS, pin_memory=True)
    cv_vl_ld  = DataLoader(cv_vl_ds, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True)

    # fresh model + optimiser
    fold_model = LensFinderEfficientNet().to(device)
    fold_opt   = optim.AdamW(fold_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    fold_sch   = optim.lr_scheduler.CosineAnnealingWarmRestarts(fold_opt, T_0=10, eta_min=1e-6)
    fold_scl   = GradScaler()
    fold_crit = FocalLoss(gamma=FOCAL_GAMMA)

    best_fold_auc, patience_cv, best_fold_wts = 0.0, 0, None

    for epoch in range(1, CV_EPOCHS + 1):
        fold_model.train()
        for imgs, lbls in cv_tr_ld:
            imgs, lbls = imgs.to(device), lbls.to(device)
            fold_opt.zero_grad()
            with autocast():
                loss = fold_crit(fold_model(imgs), lbls)
            fold_scl.scale(loss).backward()
            fold_scl.step(fold_opt); fold_scl.update()
        fold_sch.step()

        fold_model.eval()
        vp, vl = [], []
        with torch.no_grad():
            for imgs, lbls in cv_vl_ld:
                with autocast():
                    logits = fold_model(imgs.to(device))
                vp.extend(torch.sigmoid(logits).cpu().numpy())
                vl.extend(lbls.numpy())
        v_auc = roc_auc_score(vl, vp)
        if v_auc > best_fold_auc:
            best_fold_auc  = v_auc
            best_fold_wts  = {k: v.clone() for k, v in fold_model.state_dict().items()}
            patience_cv    = 0
        else:
            patience_cv += 1
            if patience_cv >= CV_PATIENCE:
                print(f'  Early stop ep={epoch}')
                break

    # OOF predictions
    fold_model.load_state_dict(best_fold_wts)
    fold_model.eval()
    oof_p = []
    with torch.no_grad():
        for imgs, _ in cv_vl_ld:
            with autocast():
                logits = fold_model(imgs.to(device))
            oof_p.extend(torch.sigmoid(logits).cpu().numpy())
    oof_scores_arr[val_idx_cv] = oof_p

    # metrics
    f_roc = roc_auc_score(np.array(vl_l_cv), oof_p)
    f_pr  = average_precision_score(np.array(vl_l_cv), oof_p)
    f_tpr, _ = tpr_at_fpr(np.array(vl_l_cv), oof_p, 0.01)
    fold_roc_list.append(f_roc); fold_pr_list.append(f_pr); fold_tpr_list.append(f_tpr)

    torch.save(best_fold_wts, f'/kaggle/working/fold_{fold+1}.pth')
    print(f'  ROC-AUC={f_roc:.4f}  PR-AUC={f_pr:.4f}  TPR@1%FPR={f_tpr:.4f}')

# OOF aggregate
oof_roc = roc_auc_score(oof_labels_arr, oof_scores_arr)
oof_pr  = average_precision_score(oof_labels_arr, oof_scores_arr)
oof_tpr, _ = tpr_at_fpr(oof_labels_arr, oof_scores_arr, 0.01)

print(f'\n{"="*65}')
print(f'  5-FOLD CV SUMMARY')
print(f'  Per-fold ROC-AUC  : {np.mean(fold_roc_list):.4f} ± {np.std(fold_roc_list):.4f}')
print(f'  Per-fold PR-AUC   : {np.mean(fold_pr_list):.4f} ± {np.std(fold_pr_list):.4f}')
print(f'  Per-fold TPR@1%   : {np.mean(fold_tpr_list):.4f} ± {np.std(fold_tpr_list):.4f}')
print(f'  OOF  ROC-AUC      : {oof_roc:.4f}')
print(f'  OOF  PR-AUC       : {oof_pr:.4f}')
print(f'  OOF  TPR@1%FPR    : {oof_tpr:.4f}')
print(f'{"="*65}')
